# 6 — Customising the Characterisation Matrix
In this notebook we look inside the characterisation step of an LCA calculation and build a **custom elementwise characterisation matrix**. Standard Brightway characterisation applies a single factor to each elementary flow regardless of which process emitted it. Here we build a sparse *(flows × processes)* characterisation matrix where the factor can vary by process.

## Using LLMs in this notebook
Use Claude, ChatGPT, or Copilot freely. Good prompts:
- *"In scipy.sparse, what is the COO format and how do I convert a CSR matrix to it?"*
- *"If I have a sparse COO matrix with arrays row, col, data, how do I index a numpy array by row to get one value per non-zero?"*
- *"What does A.multiply(B) do in scipy.sparse, and how is it different from A @ B?"*
- *"In bw2calc, what does lca.dicts.activity contain and how do I look up a column index from a node id?"*

## The LCA equations
The classic matrix LCA equation is:
$$h = C B A^{-1} f$$
where:
- $f$ — **demand vector** (products)
- $A$ — **technosphere matrix** (products × processes)
- $B$ — **biosphere matrix** (elementary flows × processes)
- $C$ — **characterisation matrix** (impact categories × elementary flows)
- $h$ — **impact score(s)**

### Why Brightway uses $h = CB \cdot \text{diag}(A^{-1}f)$
Solving $s = A^{-1}f$ gives the **supply array** — how much each process must run. Rather than computing the dense product $Bs$ (collapsing to a vector), Brightway forms the **inventory** as $B \cdot \text{diag}(s)$, keeping a full *(elementary flows × processes)* matrix. This:
1. Preserves **process contribution information** — you can see which process drove each emission
2. Is cheaper — $\text{diag}(s)$ just scales columns, no full matrix multiply

### Matrix and array shapes
| Name | Symbol | Shape | Meaning |
|---|---|---|---|
| Technosphere matrix | $A$ | products × processes | how products link to processes |
| Demand vector | $f$ | products | what you want |
| Biosphere matrix | $B$ | flows × processes | emissions per unit of each process |
| Characterisation matrix | $C$ | flows × flows (diagonal) | impact factor per flow |
| Supply array | $s = A^{-1}f$ | processes | how much each process runs |
| Inventory | $B \cdot \text{diag}(s)$ | flows × processes | attributed emissions |
| Characterised inventory | $C \cdot B \cdot \text{diag}(s)$ | flows × processes | attributed impacts |
| Score | $\sum$ characterised inventory | scalar | total impact |

## The elementwise characterisation approach

Standard LCA: flow $i$ always gets characterisation factor $CF_i$, regardless of which process emitted it.

**Elementwise LCA** builds a characterisation matrix with the same shape as the inventory *(flows × processes)*, where entry $(i, j)$ is the effective CF for flow $i$ when emitted by process $j$:

$$\text{char\_matrix}[i, j] = CF_{ij}$$

The score is then a single elementwise step:

$$\text{score} = \sum_{i,j} \text{char\_matrix}[i,j] \times \text{inventory}[i,j]$$

In code: `characterized_inventory = char_matrix.multiply(lca.inventory)` then `float(characterized_inventory.sum())`

### Building the characterisation matrix from the COO format

The inventory is sparse — most entries are zero. We only need CFs where there are actual emissions. The COO (**coordinate**) format of a sparse matrix exposes these positions directly as three arrays:

```python
inv_coo = lca.inventory.tocoo()
# inv_coo.row[k]  — row index of the k-th non-zero (which flow)
# inv_coo.col[k]  — column index of the k-th non-zero (which process)
# inv_coo.data[k] — value of the k-th non-zero
```

To build the standard characterisation matrix (same CF for all processes):

```python
cf_values = cf_vector[inv_coo.row]   # CF_i for each non-zero at row i
char_matrix = sp.coo_matrix(
    (cf_values, (inv_coo.row, inv_coo.col)),
    shape=lca.inventory.shape
)
characterized_inventory = char_matrix.multiply(lca.inventory)
score = float(characterized_inventory.sum())
```

To customise: change `cf_values[k]` for specific positions $(i, j)$ before building the matrix.

### Three applications in this notebook

| Application | `char_matrix[i, j]` |
|---|---|
| Scandinavian weighting | $2 \times CF_i$ if process $j$ is in NO/SE/DK/FI, else $CF_i$ |
| Geometric land use | $CF_{\text{base}} \times L_c^{\alpha-1}$ where $c$ = country of process $j$ |
| Renewables bonus | $0.1 \times CF_i$ for wind/solar process $j$, else $CF_i$ |

## Setup

We use the BAFU-2025-v2 Swiss LCI database (11,747 processes) and the Ecological Scarcity 2021 LCIA family. We also pre-build a `node_location` dictionary that maps each node's integer id to its location string — this avoids making a separate database query per process inside loops later on.

In [ ]:
import bw2data as bd
import bw2calc as bc
import numpy as np
import scipy.sparse as sp

bd.projects.set_current('BAFU 2025 v2')
db = bd.Database('BAFU-2025-v2')
print(f'Database: {db.name}, {len(db)} processes')
print(f'LCIA methods: {len(list(bd.methods))}')

# Pre-build a location cache — one pass over the database instead of one query per loop iteration
node_location = {n.id: n.get('location', 'GLO') for n in db}
print(f'Location cache: {len(node_location)} entries')

## Verifying the approach

Before modifying anything we run a standard LCA to establish a baseline. We will then reconstruct the same score using our custom characterisation matrix — confirming that the two approaches are equivalent when no modifications are applied.

In [ ]:
nat_gas_no = bd.get_node(
    database='BAFU-2025-v2',
    name='Natural gas, at production offshore',
    location='NO',
)

functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    demand={nat_gas_no: 1},
    method=('Ecological Scarcity 2021', 'Global warming'),
    remapping=False,
)
lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca.lci()
lca.lcia()

print(f'Product: {nat_gas_no["name"]} ({nat_gas_no["location"]})')
print(f'Standard GW score: {lca.score:.4f} UBP')
print()
print('Matrix dimensions:')
print(f'  Technosphere A:          {lca.technosphere_matrix.shape}  (products x processes)')
print(f'  Biosphere B:             {lca.biosphere_matrix.shape}  (flows x processes)')
print(f'  Characterisation C:      {lca.characterization_matrix.shape}  (flows x flows, diagonal)')
print(f'  Supply array s:          {lca.supply_array.shape}  (processes)')
print(f'  Inventory B*diag(s):     {lca.inventory.shape}  (flows x processes)')
print(f'  Characterised inventory: {lca.characterized_inventory.shape}')

### Step 1 — Extract the characterisation factor vector

`lca.characterization_matrix` is a diagonal *(flows × flows)* sparse matrix. Summing along axis 1 gives one value per row — for a diagonal matrix this is just the diagonal itself. The result is a dense array `cf_vector` of length *n_flows*, with 0 for flows that have no characterisation factor under this method.

In [ ]:
cf_vector = np.array(lca.characterization_matrix.sum(axis=1)).ravel()
n_flows, n_processes = lca.inventory.shape

print(f'Flows in inventory:         {n_flows}')
print(f'Non-zero CFs (characterised): {(cf_vector != 0).sum()}')
print(f'Processes in supply chain:  {n_processes}')

### Step 2 — Convert the inventory to COO format

The inventory matrix is very sparse — the vast majority of *(flow, process)* pairs have zero emissions. The **COO (coordinate) format** represents a sparse matrix as three parallel arrays:

| Array | Length | Contains |
|---|---|---|
| `row` | nnz | row index (which flow) of each non-zero |
| `col` | nnz | column index (which process) of each non-zero |
| `data` | nnz | the emission value at that position |

This gives us direct access to every emission's *(flow, process)* position — exactly what we need to assign process-specific characterisation factors.

In [ ]:
inv_coo = lca.inventory.tocoo()

total_possible = n_flows * n_processes
fill_pct = 100 * inv_coo.nnz / total_possible
print(f'Non-zero entries: {inv_coo.nnz:,}  out of  {total_possible:,} possible  ({fill_pct:.3f}% fill)')
print(f'inv_coo.row shape: {inv_coo.row.shape}  — one entry per non-zero')
print(f'inv_coo.col shape: {inv_coo.col.shape}')
print(f'inv_coo.data shape: {inv_coo.data.shape}')

### Step 3 — Build the characterisation matrix and verify

For each non-zero at position *(i, j)* we assign `cf_vector[i]` — the standard CF for that flow. Because we index `cf_vector` by `inv_coo.row`, this is a single vectorised numpy operation with no Python loop. The resulting `char_matrix` has the same sparsity pattern as the inventory and produces the standard LCA score when multiplied elementwise.

**A small example.** Suppose the inventory has 3 flows and 2 processes, with non-zeros at three positions:

```
inv_coo.row  = [0,  2,  2]      ← which flow
inv_coo.col  = [0,  0,  1]      ← which process
inv_coo.data = [1.5, 0.8, 2.1]  ← emission value
```

and `cf_vector = [10, 0, 30]` (flow 0 → CF 10, flow 1 → uncharacterised, flow 2 → CF 30). Then:

```
cf_vector[inv_coo.row]
  = cf_vector[[0, 2, 2]]
  = [10, 30, 30]
```

`char_matrix` is built with value 10 at *(0, 0)*, 30 at *(2, 0)*, 30 at *(2, 1)*. Multiplying elementwise by the inventory and summing:

```
10 × 1.5  +  30 × 0.8  +  30 × 2.1  =  15 + 24 + 63  =  102
```

The key trick: indexing a numpy array with another array (`cf_vector[inv_coo.row]`) broadcasts the lookup across every non-zero in one step, with no explicit loop over positions.

In [ ]:
cf_values = cf_vector[inv_coo.row]   # CF_i at each non-zero position

char_matrix = sp.coo_matrix(
    (cf_values, (inv_coo.row, inv_coo.col)),
    shape=lca.inventory.shape
)

# Elementwise multiply: char_matrix[i,j] × inventory[i,j] = CF_i × emission
characterized_inventory = char_matrix.multiply(lca.inventory)
score_elementwise = float(characterized_inventory.sum())

print(f'Standard score:    {lca.score:.6f} UBP')
print(f'Elementwise score: {score_elementwise:.6f} UBP')
print(f'Match: {np.isclose(lca.score, score_elementwise)}')
print(f'\nchar_matrix entries: {char_matrix.nnz}  (same sparsity as inventory)')

## Exercise 1 — Scandinavian processes get a 2× characterisation factor

Standard GW characterisation assigns the same factor to CO₂ regardless of where it is emitted. As a sensitivity test, suppose we apply a **2× factor to all processes located in Scandinavia** (Norway, Sweden, Denmark, Finland).

**Your task:**
1. Run a GW LCA for each product (you can reuse `lca_ex1` by calling `lca_ex1.lci(demand={product.id: 1})`)
2. Extract `cf_vector` from `lca_ex1.characterization_matrix`
3. Convert the inventory to COO format: `inv_coo = lca_ex1.inventory.tocoo()`
4. Build `cf_values = cf_vector[inv_coo.row].copy()` — standard CFs for every non-zero
5. For each non-zero at column `j`, if process `j` is Scandinavian: `cf_values[k] *= 2.0`
6. Build `char_matrix = sp.coo_matrix((cf_values, (inv_coo.row, inv_coo.col)), shape=...)`
7. Score: `float(char_matrix.multiply(lca_ex1.inventory).sum())`
8. Report the standard vs modified score for each product, and explain why the Danish product changes more

**Key lookup:** build `col_to_loc = {col: node_location.get(nid, 'GLO') for nid, col in lca.dicts.activity.items()}` to map column indices to locations.

In [ ]:
dk_electricity = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, low voltage, production DK, at grid',
    location='DK',
)
ch_electricity = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, low voltage, certified electricity, at grid',
    location='CH',
)

### Why these two products?

Danish low-voltage electricity is produced from a mix of wind, gas, and Nordic grid imports. A significant share of its GW supply chain — roughly 25% — comes from Scandinavian processes (natural gas from Norwegian offshore production, electricity from Sweden).

Swiss certified electricity (mainly hydro and nuclear) has essentially no Scandinavian processes in its supply chain. Doubling the Scandinavian CF should therefore affect the Danish product much more than the Swiss one — a good test of the technique.

In [ ]:
functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    demand={dk_electricity: 1},
    method=('Ecological Scarcity 2021', 'Global warming'),
    remapping=False,
)
lca_ex1 = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca_ex1.lci()
lca_ex1.lcia()
print(f'Danish electricity — standard GW: {lca_ex1.score:.2f} UBP')

lca_ex1.lci(demand={ch_electricity.id: 1})
lca_ex1.lcia()
print(f'Swiss electricity  — standard GW: {lca_ex1.score:.2f} UBP')

SCANDINAVIAN_LOCATIONS = {'NO', 'SE', 'DK', 'FI'}

In [ ]:
# Exercise 1 — your code here

# Hint: for each product, re-run the LCA and build the characterisation matrix.
# Re-run pattern (data_objs are already set up from the first call above):
#
#   lca_ex1.lci(demand={product.id: 1})
#   lca_ex1.lcia()
#
# Then build the characterisation matrix:
#
#   cf_vector = np.array(lca_ex1.characterization_matrix.sum(axis=1)).ravel()
#   inv_coo = lca_ex1.inventory.tocoo()
#   cf_values = cf_vector[inv_coo.row].copy()   # start: standard CF_i everywhere
#
#   # Map column index → location
#   col_to_loc = {col: node_location.get(nid, 'GLO')
#                 for nid, col in lca_ex1.dicts.activity.items()}
#
#   # Double the CF for Scandinavian process columns
#   is_scand = np.array([col_to_loc.get(col, 'GLO') in SCANDINAVIAN_LOCATIONS
#                        for col in inv_coo.col])
#   cf_values[is_scand] *= 2.0
#
#   char_matrix = sp.coo_matrix(
#       (cf_values, (inv_coo.row, inv_coo.col)), shape=lca_ex1.inventory.shape
#   )
#   characterized_inventory = char_matrix.multiply(lca_ex1.inventory)
#   modified_score = float(characterized_inventory.sum())

## Example: geometric land use scaling

Standard LCA is **linear**: 2 m²·yr of land occupation in a country has exactly twice the impact of 1 m²·yr. Ecology suggests otherwise — a single large monoculture destroys more habitat connectivity and biodiversity than the same total area split across many small sites.

One way to model this is to make the characterisation factor **increase with the total land use attributed to a country**:

$$\text{impact}_c = L_c^{\,\alpha} \times CF_{\text{base}}$$

where $L_c$ is the total land occupation attributed to country $c$ (m²·yr) and $\alpha > 1$. For $\alpha = 1$ this is standard linear LCA.

### From country totals to the characterisation matrix

For each non-zero $(i, j)$ in the inventory, we set:

$$\text{char\_matrix}[i, j] = CF_{\text{base}} \times L_c^{\,\alpha-1}$$

where $c$ is the country of process $j$. Summing over all processes in country $c$:

$$\sum_{j \in c} L_j \times CF_{\text{base}} \times L_c^{\alpha-1} = L_c \times L_c^{\alpha-1} \times CF_{\text{base}} = L_c^{\,\alpha} \times CF_{\text{base}} \checkmark$$

### Concentrated vs distributed land use

| Product | Land use | Standard score | Geometric ($\alpha=1.5$) |
|---|---|---|---|
| Concentrated (all 2.92 m²·yr in one country) | 2.92^1 × 520 = **1518 UBP** | same | 2.92^1.5 × 520 = **2596 UBP** |
| Distributed (0.73 m²·yr each in four countries) | 4 × 0.73 × 520 = **1518 UBP** | same | 4 × 0.73^1.5 × 520 = **1304 UBP** |

Standard LCA says both are **identical**. Geometric LCA says the concentrated product is **2× worse** — with a modest exponent of just 1.5.

In [ ]:
method_lu = bd.Method(('Ecological Scarcity 2021', 'Land use'))
lu_flow_ids = {key for key, _ in method_lu.load()}   # set of characterised flow IDs

rape_ch = bd.get_node(database='BAFU-2025-v2', name='Rape seed IP, at farm', location='CH')

functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    demand={rape_ch: 1},
    method=('Ecological Scarcity 2021', 'Land use'),
    remapping=False,
)
lca_lu = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca_lu.lci()
lca_lu.lcia()
print(f'Product: {rape_ch["name"]} ({rape_ch["location"]})')
print(f'Standard land use score: {lca_lu.score:.1f} UBP')

# Row indices in the inventory that correspond to land use flows
lu_rows = [
    lca_lu.dicts.biosphere[fid]
    for fid in lu_flow_ids
    if fid in lca_lu.dicts.biosphere
]
print(f'Land use flow types present in inventory: {len(lu_rows)}')

### Summing land use by country

We sum raw land use (m²·yr) across all land-use flow rows for each process column, then group by country. This gives us $L_c$ — the total land occupation attributed to each country in this product system — which is the quantity we will raise to the power $\alpha$.

In [ ]:
# Raw land use (m2.yr) per process: sum across all land use flow types
raw_lu_per_process = np.array(lca_lu.inventory[lu_rows, :].sum(axis=0)).ravel()

country_lu = {}
for col_idx in np.where(raw_lu_per_process > 1e-9)[0]:
    node_id = lca_lu.dicts.activity.reversed[col_idx]
    country = node_location.get(node_id, 'GLO')
    country_lu[country] = country_lu.get(country, 0) + raw_lu_per_process[col_idx]

total_lu = sum(country_lu.values())
print(f'Total raw land use: {total_lu:.4f} m2.yr')
print('By country (>0.1% share only):')
for country, lu in sorted(country_lu.items(), key=lambda x: -x[1]):
    share = 100 * lu / total_lu
    if share > 0.1:
        print(f'  {country:6}: {lu:.4f} m2.yr  ({share:.1f}%)')

### Building the geometric characterisation matrix

For each non-zero *(i, j)* in the inventory we set:

$$\text{char\_matrix}[i, j] = CF_{\text{base}} \times L_c^{\,\alpha - 1}$$

where $c$ is the country of process $j$. We first build `cf_base` — the base CF for every inventory row (520 UBP/m²·yr for land use flows, 0 for everything else). Then we compute `geo_factor[k]` = $L_c^{\alpha-1}$ for the country of the $k$-th non-zero's process column, vectorised across all non-zeros at once.

In [ ]:
CF_annual_crop = 520   # UBP per m2.yr — dominant flow type for rapeseed
alpha = 1.5

# Column index → location for all processes in this LCA
col_to_loc = {
    col: node_location.get(nid, 'GLO')
    for nid, col in lca_lu.dicts.activity.items()
}

inv_coo = lca_lu.inventory.tocoo()

# Base CF per inventory row: 520 for land use flows, 0 for all others
cf_base = np.zeros(lca_lu.inventory.shape[0])
for row in lu_rows:
    cf_base[row] = CF_annual_crop

# For each non-zero, look up L_c for its process column's country
L_c_per_nz = np.array([
    country_lu.get(col_to_loc.get(col, 'GLO'), 0.0)
    for col in inv_coo.col
])
geo_factor = np.where(L_c_per_nz > 0, L_c_per_nz ** (alpha - 1), 1.0)

# Final CF for each non-zero: CF_base[i] × L_c^(alpha-1)
cf_values_lu = cf_base[inv_coo.row] * geo_factor

char_matrix_lu = sp.coo_matrix(
    (cf_values_lu, (inv_coo.row, inv_coo.col)),
    shape=lca_lu.inventory.shape
)
characterized_inventory = char_matrix_lu.multiply(lca_lu.inventory)
geometric_score = float(characterized_inventory.sum())

standard_approx = total_lu * CF_annual_crop
print(f'Standard score (approx, CF=520):  {standard_approx:.1f} UBP')
print(f'Geometric score (alpha={alpha}):     {geometric_score:.1f} UBP')
print(f'Amplification factor:             {geometric_score / standard_approx:.2f}x')

### Concentrated vs distributed: comparing the two approaches

Swiss rapeseed has almost all its land use concentrated in Switzerland (~99.8%). To show what the geometric approach does, we construct a hypothetical version of the same product with the same *total* land use but spread equally across four countries. Standard LCA scores them identically; geometric LCA penalises the concentrated one.

In [ ]:
n_countries = 4
lu_each = total_lu / n_countries
distributed_countries = ['CH', 'DE', 'FR', 'PL']

standard_conc  = total_lu * CF_annual_crop
standard_dist  = total_lu * CF_annual_crop
geometric_conc = geometric_score
geometric_dist = n_countries * (lu_each ** alpha) * CF_annual_crop

print(f'Hypothetical distributed product: {lu_each:.3f} m2.yr each in {distributed_countries}')
print()
print(f'{"Product":<35} {"Standard":>10} {"Geometric (a=1.5)":>18}')
print('-' * 65)
print(f'{"Concentrated (2.92 m2.yr in CH)":<35} {standard_conc:>10.0f} {geometric_conc:>18.0f}')
print(f'{"Distributed (0.73 in each of 4)":<35} {standard_dist:>10.0f} {geometric_dist:>18.0f}')
print()
print(f'Standard LCA: both products score identically ({standard_conc:.0f} UBP)')
print(f'Geometric LCA: concentrated is {geometric_conc/geometric_dist:.1f}x worse than distributed')

## Exercise 2 — Wind and solar electricity reduce impacts

Wind turbines and solar panels have small but non-zero GHG emissions from manufacturing (steel, copper, glass fibre, silicon). Standard LCA characterises these the same as any other CO₂.

A **renewables bonus** scenario argues that wind and solar electricity production processes should receive a discount on their GW characterisation, since they displace fossil fuels over their operational lifetime.

**Your task:**
1. Find all wind-farm electricity processes in BAFU (search for `'wind farm'` in the process name)
2. Find all photovoltaic electricity processes (search for `'production mix photovoltaic'`)
3. Run a GW LCA for:
   - **Wind electricity**: `'Electricity, at 100MW wind farm, onshore, 2.3MW turbines'` (RER)
   - **Gas-fired CHP electricity**: `'Electricity, natural gas, at CHP power plant {SE} U - SE'` (SE)
4. Build `cf_values = cf_vector[inv_coo.row].copy()`; set `cf_values[k] *= 0.1` where `inv_coo.col[k]` is a renewable process
5. Build `char_matrix = sp.coo_matrix((cf_values, (inv_coo.row, inv_coo.col)), shape=...)`
6. Score: `float(char_matrix.multiply(lca.inventory).sum())`
7. Report standard vs modified scores for both products

**Expected behaviour:** the wind farm benefits from the renewable discount (its supply chain includes other wind processes); the gas CHP score is unchanged.

**Hint:** use `lca.dicts.activity.get(node_id)` — it returns `None` if a node is not in the current LCA's activity mapping, so you can skip missing nodes safely.

In [ ]:
wind_farm = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, at 100MW wind farm, onshore, 2.3MW turbines',
    location='RER',
)
gas_se = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, natural gas, at CHP power plant {SE} U - SE',
    location='SE',
)

### Finding renewable electricity processes in the database

We search the entire database for wind-farm and photovoltaic electricity processes by name and collect their node IDs into a set. When we then build the characterisation matrix for a specific product, only the IDs that appear in *that* LCA's activity dictionary — i.e. those actually present in the supply chain — will affect the score.

In [ ]:
renewable_ids = {
    n.id for n in db
    if (
        ('wind farm' in n['name'].lower() and 'electricity' in n['name'].lower())
        or 'production mix photovoltaic' in n['name'].lower()
    )
    and not n['name'].startswith('xx')
}
print(f'Renewable electricity processes found: {len(renewable_ids)}')

functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    demand={wind_farm: 1},
    method=('Ecological Scarcity 2021', 'Global warming'),
    remapping=False,
)
lca_ex2 = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca_ex2.lci()
lca_ex2.lcia()
print(f'\nWind electricity — standard GW: {lca_ex2.score:.4f} UBP/kWh')

lca_ex2.lci(demand={gas_se.id: 1})
lca_ex2.lcia()
print(f'Gas CHP (SE)      — standard GW: {lca_ex2.score:.4f} UBP/kWh')

In [ ]:
# Exercise 2 — your code here

# Hint: re-run the LCA for each product, then build the characterisation matrix.
# Re-run pattern:
#
#   lca_ex2.lci(demand={product.id: 1})
#   lca_ex2.lcia()
#
# Then:
#   cf_vector = np.array(lca_ex2.characterization_matrix.sum(axis=1)).ravel()
#   inv_coo = lca_ex2.inventory.tocoo()
#   cf_values = cf_vector[inv_coo.row].copy()
#
#   # Column indices that are renewable in this LCA
#   renew_cols = {lca_ex2.dicts.activity[nid] for nid in renewable_ids
#                if nid in lca_ex2.dicts.activity}
#
#   # Discount CF for renewable columns
#   is_renew = np.array([col in renew_cols for col in inv_coo.col])
#   cf_values[is_renew] *= 0.1
#
#   char_matrix = sp.coo_matrix(
#       (cf_values, (inv_coo.row, inv_coo.col)), shape=lca_ex2.inventory.shape
#   )
#   characterized_inventory = char_matrix.multiply(lca_ex2.inventory)
#   modified_score = float(characterized_inventory.sum())

## Summary

The **elementwise characterisation** pattern introduced in this notebook builds a sparse *(flows × processes)* characterisation matrix — where `char_matrix[i, j]` is the effective CF for flow $i$ when emitted by process $j$ — and then scores in one step:

```python
characterized_inventory = char_matrix.multiply(lca.inventory)
score = float(characterized_inventory.sum())
```

**Building the matrix:** convert the inventory to COO format to get the positions of all non-zeros, assign CFs to those positions, then construct the sparse matrix:

```python
inv_coo = lca.inventory.tocoo()                     # positions of all emissions
cf_values = cf_vector[inv_coo.row].copy()           # standard CF_i at each position
# ... modify cf_values[k] for specific (flow, process) combinations ...
char_matrix = sp.coo_matrix(
    (cf_values, (inv_coo.row, inv_coo.col)),
    shape=lca.inventory.shape
)
score = float(char_matrix.multiply(lca.inventory).sum())
```

Setting `cf_values = cf_vector[inv_coo.row]` (no modification) recovers the standard LCA score exactly.

| Exercise | Modification to `cf_values[k]` |
|---|---|
| Scandinavian weighting | `*= 2.0` where `col_to_loc[inv_coo.col[k]]` is in NO/SE/DK/FI |
| Geometric land use | `*= L_c^(α-1)` where $c$ = country of process `inv_coo.col[k]` |
| Renewables bonus | `*= 0.1` where `inv_coo.col[k]` is a wind/solar process |

**Key implementation notes:**
- `char_matrix` has the same sparsity as `lca.inventory` — non-zero only where there are emissions
- `col_to_loc = {col: node_location.get(nid, 'GLO') for nid, col in lca.dicts.activity.items()}` maps column indices to locations in O(n) from the pre-built cache
- Build the **node location cache** (`{n.id: n.get('location', 'GLO') for n in db}`) once at startup — one pass over the database, not one SQLite call per node
- Use `lca.dicts.activity.get(node_id)` (not `[]`) so missing nodes return `None` instead of raising `KeyError`